In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By


print("正在启动 Edge 浏览器...")
driver = webdriver.Edge()

print("正在访问网页...")
driver.get("http://xinpi.jxc.cnstock.com/p/xg")

print("访问成功！浏览器标题是：", driver.title)

# Get the HTML page source
html_source = driver.page_source


from bs4 import BeautifulSoup

# 初始化解析器
soup = BeautifulSoup(html_source, 'html.parser')

# 核心：提取新股日历（仅日期+星期+各动作股票，无链接/无今日标注）
stock_calendar = []
# 定位新股日历主容器
calendar_wrap = soup.find('div', class_='stock-cleander')
# 遍历每一天的日历卡片
for day_card in calendar_wrap.find_all('div', class_='cleander-wrap'):
    day_data = {}
    # 提取日期和星期
    day_time = day_card.find('div', class_='day-time')
    week = day_time.find('span').text.strip()  # 周一/周二
    year = day_time.find('p', class_='s').text.replace(week, '').strip()  # 2026
    date = day_time.find('p', class_='b').text.strip()  # 02.02
    day_data['日期'] = f'{year}-{date}'
    day_data['星期'] = week
    
    # 提取申购/上市/中签号/中签率/缴款的股票名称（无链接，多个用顿号分隔）
    table = day_card.find('table', class_='ax-table')
    for tr in table.find_all('tr'):
        td_list = tr.find_all('td')
        if len(td_list) < 2:
            continue
        action = td_list[0].span.text.strip()  # 申购/上市/中签号/中签率/缴款
        # 提取股票名称，无则为"无"，多个拼接
        stock_names = [a.text.strip() for a in td_list[1].find_all('a')]
        day_data[action] = '、'.join(stock_names) if stock_names else '无'
    
    stock_calendar.append(day_data)

# 格式化打印结果（清晰易读）
print("===== 新股日历 =====")
for day in stock_calendar:
    print(f"\n{day['日期']}（{day['星期']}）")
    print(f"  申购：{day['申购']}")
    print(f"  上市：{day['上市']}")
    print(f"  中签号公布：{day['中签号']}")
    print(f"  中签率公布：{day['中签率']}")
    print(f"  缴款：{day['缴款']}")


正在启动 Edge 浏览器...
正在访问网页...
访问成功！浏览器标题是： 中国资本市场法定信息披露平台 | 上海证券报·中国证券网
===== 新股日历 =====

2026-02.02（周一）
  申购：爱得科技、易思维
  上市：无
  中签号公布：无
  中签率公布：林平发展、电科蓝天
  缴款：爱得科技

2026-02.03（周二）
  申购：无
  上市：N世盟
  中签号公布：林平发展、电科蓝天
  中签率公布：易思维
  缴款：林平发展、电科蓝天

2026-02.04（周三）
  申购：海圣医疗
  上市：无
  中签号公布：易思维
  中签率公布：无
  缴款：海圣医疗、易思维

2026-02.05（周四）
  申购：无
  上市：无
  中签号公布：无
  中签率公布：无
  缴款：无

2026-02.06（周五）
  申购：无
  上市：无
  中签号公布：无
  中签率公布：无
  缴款：无
